In [ ]:
from google.colab import files
uploaded = files.upload()

In [ ]:
!pip install -q transformers==4.40.0 peft==0.10.0 accelerate==0.29.3

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.6/137.6 kB 9.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.0/9.0 MB 47.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 199.1/199.1 kB 18.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 297.6/297.6 kB 28.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 32.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 112.2 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sentence-transformers 5.4.1 requires transformers<6.0.0,>=4.41.0, but you have transformers 4.40.0 which is incompatible.


In [ ]:
import json, zipfile, os, re, torch
from transformers import T5ForConditionalGeneration, AutoTokenizer
from peft import PeftModel


CHECKPOINT_ZIP = "./checkpoint-3032.zip"
ADAPTER_DIR    = "./checkpoint-3032"
MODEL_NAME     = "google/flan-t5-xl"
MAX_INPUT      = 256
MAX_TARGET     = 128


if not os.path.exists(ADAPTER_DIR):
    with zipfile.ZipFile(CHECKPOINT_ZIP, 'r') as z:
        z.extractall(ADAPTER_DIR)
    print("Extracted adapter to:", ADAPTER_DIR)


dtype = (
    torch.bfloat16
    if torch.cuda.is_available() and torch.cuda.get_device_capability()[0] >= 8
    else torch.float16
)
print(f"Loading {MODEL_NAME} ({dtype})...")
base_model = T5ForConditionalGeneration.from_pretrained(
    MODEL_NAME, torch_dtype=dtype, device_map="auto"
)
print("Loading LoRA weights...")
model = PeftModel.from_pretrained(base_model, ADAPTER_DIR, is_trainable=False)
model.eval()
model.config.use_cache = True

try:
    tokenizer = AutoTokenizer.from_pretrained(ADAPTER_DIR)
except Exception:
    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

print("✅ Model ready!")

Extracted adapter to: ./checkpoint-3032
Loading google/flan-t5-xl (torch.float16)...


/usr/local/lib/python3.12/dist-packages/huggingface_hub/file_download.py:949: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json: 0.00B [00:00, ?B/s]

model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00001-of-00002.safetensors:   0%|          | 0.00/9.45G [00:00<?, ?B/s]

...

Loading checkpoint shards:   0%|          | 0/2 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

Loading LoRA weights...


Special tokens have been added in the vocabulary, make sure the associated word embeddings are fine-tuned or trained.
You set `add_prefix_space`. The tokenizer needs to be converted from the slow tokenizers


✅ Model ready!


In [ ]:

def fix_sparql_spacing(sparql):
    sparql = re.sub(r'(\S)\?', r'\1 ?', sparql)
    sparql = re.sub(r'(SELECT|DISTINCT|WHERE|FILTER|LIMIT|ORDER|BY|ASK|OPTIONAL|UNION)(\S)', r'\1 \2', sparql, flags=re.IGNORECASE)
    sparql = re.sub(r'(\S)\.(\S)', r'\1 . \2', sparql)
    sparql = re.sub(r' +', ' ', sparql).strip()
    return sparql

def fix_and_add_braces(sparql):
    sparql = fix_sparql_spacing(sparql)
    if '{' not in sparql and 'WHERE' in sparql.upper():
        match = re.search(r'(WHERE\s*)(.*?)(\s*)$', sparql, re.IGNORECASE | re.DOTALL)
        if match:
            sparql = sparql[:match.start(2)] + '{ ' + match.group(2).strip() + ' }'
    return sparql

# ── Build prompt (same format as training) ────────────────────
def build_prompt(question, dataset="DBpedia"):
    """
    Mirrors the input format from combined_output.json:
    'generate sparql [DBpedia]: <question>'
    """
    return f"generate sparql [{dataset}]: {question}"

# ── Generate SPARQL ───────────────────────────────────────────
def generate_sparql(question, dataset="DBpedia"):
    prompt = build_prompt(question, dataset)
    inp = tokenizer(
        prompt,
        return_tensors="pt",
        max_length=MAX_INPUT,
        truncation=True,
    )
    inp = {k: v.to(model.device) for k, v in inp.items()}
    with torch.no_grad():
        out = model.generate(
            **inp,
            max_new_tokens=MAX_TARGET,
            num_beams=4,
            early_stopping=True,
            length_penalty=1.0,
        )
    raw = tokenizer.decode(out[0], skip_special_tokens=True)
    fixed = fix_and_add_braces(raw)
    return raw, fixed

# ── Pretty print ──────────────────────────────────────────────
def predict(question, dataset="DBpedia"):
    """
    dataset: 'DBpedia' or 'Wikidata'
    """
    print("=" * 60)
    print(f"  Question : {question}")
    print(f"  Dataset  : {dataset}")
    print("=" * 60)
    raw, fixed = generate_sparql(question, dataset)
    print(f"\n   Generated SPARQL query:")
    print(f"  {fixed}")
    print("=" * 60)
    return fixed

In [ ]:


question =  "Which  governing body of the Oahu Railway and Land Company is also the military branch of the Jimmy Quillen ?\nplan: step1: action: find_entity | surface_form: Jimmy Quillen | output_variable: ?jimmy_quillen | semantic_type: ENTITY || step2: action: find_entity | surface_form: Oahu Railway and Land Company | output_variable: ?oahu_railway_and_lan | semantic_type: ENTITY || step3: action: find_object | property: militaryBranch | subject_variable: ?jimmy_quillen | output_variable: ?uri | semantic_type: PROPERTY || step4: action: find_object | property: governingBody | subject_variable: ?oahu_railway_and_lan | semantic_type: PROPERTY | join_type: intersect | filter_variable: ?uri || step5: action: distinct | semantic_type: MODIFIER\nschema:\n  Jimmy Quillen -> dbr:Jimmy_Quillen\n  Oahu Railway and Land Company -> dbr:Oahu_Railway_and_Land_Company\n  militaryBranch -> dbo:militaryBranch\n  governingBody -> dbo:governingBody"
dataset  = "DBpedia"

sparql = predict(question, dataset)

  Question : Which  governing body of the Oahu Railway and Land Company is also the military branch of the Jimmy Quillen ?
plan: step1: action: find_entity | surface_form: Jimmy Quillen | output_variable: ?jimmy_quillen | semantic_type: ENTITY || step2: action: find_entity | surface_form: Oahu Railway and Land Company | output_variable: ?oahu_railway_and_lan | semantic_type: ENTITY || step3: action: find_object | property: militaryBranch | subject_variable: ?jimmy_quillen | output_variable: ?uri | semantic_type: PROPERTY || step4: action: find_object | property: governingBody | subject_variable: ?oahu_railway_and_lan | semantic_type: PROPERTY | join_type: intersect | filter_variable: ?uri || step5: action: distinct | semantic_type: MODIFIER
schema:
  Jimmy Quillen -> dbr:Jimmy_Quillen
  Oahu Railway and Land Company -> dbr:Oahu_Railway_and_Land_Company
  militaryBranch -> dbo:militaryBranch
  governingBody -> dbo:governingBody
  Dataset  : DBpedia

   Generated SPARQL query:
  SELECT D